# Step 9: Model Evaluation and Error Analysis

## Purpose
This notebook performs honest final evaluation of the tuned churn models and investigates where they fail.

## What this step answers
- How well do the tuned models perform on validation and test data?
- Where do the models make mistakes?
- Are the models stable across important customer segments?
- Which model should we trust more for final interpretation and deployment thinking?

## Why this step matters
A single metric can hide important weaknesses. Step 9 helps us understand not only **how good** a model is, but also **how it fails**.

## Models evaluated here
- **Logistic Regression**
- **Random Forest**

## Key rule
This is the first stage where we look at the **test set**. We do it only after model selection and tuning decisions have already been made.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Rebuild the feature set and split

We rebuild the same cleaned feature set and the same data split used in earlier modeling steps.

## Why?
This keeps final evaluation consistent with the way the models were developed. If we changed the feature logic or split here, we would not be evaluating the same modeling process anymore.

In [2]:
PROJECT_DIR = Path.cwd()
CLEANED_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv"
RANDOM_STATE = 42

df = pd.read_csv(CLEANED_FILE)

df['service_count'] = (
    (df['PhoneService'] == 'Yes').astype(int)
    + (df['MultipleLines'] == 'Yes').astype(int)
    + (df['InternetService'] != 'No').astype(int)
    + (df['OnlineSecurity'] == 'Yes').astype(int)
    + (df['OnlineBackup'] == 'Yes').astype(int)
    + (df['DeviceProtection'] == 'Yes').astype(int)
    + (df['TechSupport'] == 'Yes').astype(int)
    + (df['StreamingTV'] == 'Yes').astype(int)
    + (df['StreamingMovies'] == 'Yes').astype(int)
)

df['avg_monthly_value_from_total'] = (df['TotalCharges'] / df['tenure'].replace(0, 1)).round(2)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
df['has_long_term_contract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

X = df.drop(columns=['customerID', 'Churn']).copy()
y = df['Churn'].map({'No': 0, 'Yes': 1}).copy()

categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)

segment_df = df.loc[X.index, ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract', 'InternetService', 'PaymentMethod', 'tenure']].copy()
segment_df['tenure_group'] = pd.cut(
    segment_df['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['0-12', '13-24', '25-48', '49-72'],
)

X_train.shape, X_val.shape, X_test.shape

((4218, 23), (1407, 23), (1407, 23))

## 2. Define the tuned models

We use the tuned candidates carried forward from Step 8.

## Why?
Step 9 is about honest evaluation, not more tuning. So we freeze the model choices and now measure how they behave on validation and test data.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            C=1.0,
            class_weight='balanced',
            solver='lbfgs',
        )),
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_estimators=200,
            max_depth=8,
            min_samples_leaf=5,
            class_weight='balanced',
        )),
    ]),
}

## 3. Define evaluation helpers

We want a consistent evaluation function for both models.

## Why?
If we compute metrics differently for each model, the comparison becomes unreliable.

We also create helper functions for confusion matrix analysis and segment-based error analysis so that we can move beyond a single summary score.

In [4]:
def collect_split_metrics(model, X_data, y_data, split_name, model_name):
    y_pred = model.predict(X_data)
    y_scores = model.predict_proba(X_data)[:, 1]

    return {
        'model': model_name,
        'split': split_name,
        'accuracy': round(accuracy_score(y_data, y_pred), 4),
        'precision': round(precision_score(y_data, y_pred, zero_division=0), 4),
        'recall': round(recall_score(y_data, y_pred, zero_division=0), 4),
        'f1': round(f1_score(y_data, y_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_data, y_scores), 4),
    }


def collect_confusion_details(model, X_data, y_data, split_name, model_name):
    y_pred = model.predict(X_data)
    tn, fp, fn, tp = confusion_matrix(y_data, y_pred).ravel()

    return {
        'model': model_name,
        'split': split_name,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def build_error_frame(model, X_data, y_data, split_name, model_name, segment_slice):
    y_pred = model.predict(X_data)
    y_scores = model.predict_proba(X_data)[:, 1]

    error_frame = segment_slice.copy()
    error_frame['actual'] = y_data.values
    error_frame['predicted'] = y_pred
    error_frame['score'] = y_scores
    error_frame['correct'] = (error_frame['actual'] == error_frame['predicted']).astype(int)
    error_frame['error_type'] = 'correct'
    error_frame.loc[(error_frame['actual'] == 1) & (error_frame['predicted'] == 0), 'error_type'] = 'false_negative'
    error_frame.loc[(error_frame['actual'] == 0) & (error_frame['predicted'] == 1), 'error_type'] = 'false_positive'
    error_frame['model'] = model_name
    error_frame['split'] = split_name

    return error_frame


def summarize_segment_errors(error_frame, segment_column):
    summary = error_frame.groupby(segment_column).apply(
        lambda group: pd.Series({
            'count': len(group),
            'actual_churn_rate': round(group['actual'].mean(), 4),
            'predicted_churn_rate': round(group['predicted'].mean(), 4),
            'accuracy': round(group['correct'].mean(), 4),
            'false_negative_rate': round(((group['actual'] == 1) & (group['predicted'] == 0)).mean(), 4),
            'false_positive_rate': round(((group['actual'] == 0) & (group['predicted'] == 1)).mean(), 4),
        })
    ).reset_index()

    return summary.sort_values(by=['false_negative_rate', 'false_positive_rate'], ascending=False)

## 4. Fit both tuned models and compare validation/test performance

We fit each tuned model on the training set and evaluate on both validation and test sets.

## Why evaluate both?
- **Validation** helps us compare what we expected from Step 8
- **Test** tells us how well that tuned choice generalizes to truly held-out data

If test performance falls sharply below validation performance, that is a warning sign.

In [5]:
metrics_rows = []
confusion_rows = []
error_frames = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    metrics_rows.append(collect_split_metrics(model, X_val, y_val, 'validation', model_name))
    metrics_rows.append(collect_split_metrics(model, X_test, y_test, 'test', model_name))

    confusion_rows.append(collect_confusion_details(model, X_val, y_val, 'validation', model_name))
    confusion_rows.append(collect_confusion_details(model, X_test, y_test, 'test', model_name))

    error_frames.append(
        build_error_frame(model, X_val, y_val, 'validation', model_name, segment_df.loc[X_val.index])
    )
    error_frames.append(
        build_error_frame(model, X_test, y_test, 'test', model_name, segment_df.loc[X_test.index])
    )

final_metrics_df = pd.DataFrame(metrics_rows).sort_values(by=['split', 'recall', 'f1', 'roc_auc'], ascending=[True, False, False, False])
confusion_df = pd.DataFrame(confusion_rows)
errors_df = pd.concat(error_frames, ignore_index=True)

final_metrics_df

,model,split,accuracy,precision,recall,f1,roc_auc
1,Logistic Regression,test,0.7299,0.4949,0.7861,0.6074,0.8325
3,Random Forest,test,0.7555,0.5275,0.7701,0.6261,0.8334
0,Logistic Regression,validation,0.7512,0.5223,0.7513,0.6162,0.8350
2,Random Forest,validation,0.7719,0.5546,0.7193,0.6263,0.8362


In [6]:
confusion_df

,model,split,tn,fp,fn,tp
0,Logistic Regression,validation,776,257,93,281
1,Logistic Regression,test,733,300,80,294
2,Random Forest,validation,817,216,105,269
3,Random Forest,test,775,258,86,288


## 5. Review confusion matrix details

The confusion matrix lets us see the exact balance of:
- true positives
- false positives
- false negatives
- true negatives

## Why this matters
In churn work, false negatives are often especially costly because they are the customers we fail to catch before they leave.

## 6. Segment-based error analysis

Now we look at where the models fail by customer segment.

## Why?
A model can look good overall but still fail badly for important groups such as:
- newer customers
- customers on month-to-month contracts
- customers with certain payment methods
- customers with certain internet service types

This helps us understand model blind spots.

In [7]:
segment_columns = ['Contract', 'InternetService', 'PaymentMethod', 'tenure_group', 'SeniorCitizen']
segment_summaries = {}

for model_name in errors_df['model'].unique():
    for split_name in errors_df['split'].unique():
        subset = errors_df[(errors_df['model'] == model_name) & (errors_df['split'] == split_name)].copy()
        for segment_column in segment_columns:
            key = f"{model_name} | {split_name} | {segment_column}"
            segment_summaries[key] = summarize_segment_errors(subset, segment_column)

segment_summaries['Random Forest | test | Contract']

C:\Users\HP\AppData\Local\Temp\ipykernel_18580\2086087927.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = error_frame.groupby(segment_column).apply(
C:\Users\HP\AppData\Local\Temp\ipykernel_18580\2086087927.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = error_frame.groupby(segment_column).apply(
C:\Users\HP\AppData\Local\Temp\ipykernel_18580\2086087927.py:49: FutureWarning: DataFrameGro

,Contract,count,actual_churn_rate,predicted_churn_rate,accuracy,false_negative_rate,false_positive_rate
1,One year,290.0,0.1379,0.0207,0.8483,0.1345,0.0172
0,Month-to-month,790.0,0.4127,0.6835,0.6304,0.0494,0.3203
2,Two year,327.0,0.0245,0.0000,0.9755,0.0245,0.0000


## 7. Stability checks

We compare validation and test performance directly.

## Why?
A model that looks strong on validation but weak on test may not generalize well enough to trust.

We call a model more stable when the validation-to-test drop is small.

In [8]:
validation_metrics = final_metrics_df[final_metrics_df['split'] == 'validation'].set_index('model')
test_metrics = final_metrics_df[final_metrics_df['split'] == 'test'].set_index('model')

stability_rows = []

for model_name in validation_metrics.index:
    stability_rows.append({
        'model': model_name,
        'accuracy_drop': round(validation_metrics.loc[model_name, 'accuracy'] - test_metrics.loc[model_name, 'accuracy'], 4),
        'precision_drop': round(validation_metrics.loc[model_name, 'precision'] - test_metrics.loc[model_name, 'precision'], 4),
        'recall_drop': round(validation_metrics.loc[model_name, 'recall'] - test_metrics.loc[model_name, 'recall'], 4),
        'f1_drop': round(validation_metrics.loc[model_name, 'f1'] - test_metrics.loc[model_name, 'f1'], 4),
        'roc_auc_drop': round(validation_metrics.loc[model_name, 'roc_auc'] - test_metrics.loc[model_name, 'roc_auc'], 4),
    })

stability_df = pd.DataFrame(stability_rows)
stability_df

,model,accuracy_drop,precision_drop,recall_drop,f1_drop,roc_auc_drop
0,Logistic Regression,0.0213,0.0274,-0.0348,0.0088,0.0025
1,Random Forest,0.0164,0.0271,-0.0508,0.0002,0.0028


## 8. How to interpret Step 9 outputs

Use the notebook outputs in this order:

1. `final_metrics_df`
   - compare validation vs test performance
   - identify the strongest final candidate

2. `confusion_df`
   - inspect false negatives and false positives directly
   - ask whether the model is missing too many churners

3. `segment_summaries[...]`
   - identify customer groups where the model struggles
   - look especially at false-negative-heavy segments

4. `stability_df`
   - check whether validation and test performance are close
   - smaller drops mean more confidence in generalization

## What to bring back to me
After you run this notebook, send me:
- `final_metrics_df`
- `confusion_df`
- `stability_df`
- the segment summary table you think is most interesting

Then I will help you interpret the model failures and decide which model is truly best for final use.

# Step 9 notebook setup complete.
# Run the notebook from the top and review `final_metrics_df`, `confusion_df`, `stability_df`, and the segment summaries.